# Capstone – Transition Risk Score del sistema bancario italiano

Dati di partenza: intensità emissiva per settore (Eurostat) e prestiti per settore/regione (Banca d'Italia, BDS).
Periodo comune alle due fonti: 2011-2023.

## 1. Eurostat – intensità emissiva per settore (env_ac_aeint_r2)

Italia, gas serra su valore aggiunto lordo, prezzi concatenati 2020. Copertura dal 2008 (periodo precedente non completo secondo la stessa Eurostat) al 2023 (il 2024 presenta un numero eccessivo di settori mancanti ed è stato escluso).

In [1]:
import requests as req
import pandas as pd

url_eurostat = "https://ec.europa.eu/eurostat/api/dissemination/statistics/1.0/data/env_ac_aeint_r2"

geo = ["IT"]
time = [str(year) for year in range(2008, 2024)]
unit = "G_EUR_CLV20"
airpol = "GHG"
na_item = "B1G"  # valore aggiunto lordo

nace_r2 = ["A", "A01", "A02", "A03", "B", "C", "C10-C12", "C13-C15", "C16", "C17", "C18", "C19", "C20", "C21", "C22", "C23", "C24", "C25", "C26", "C27", "C28", "C29", "C30", "C31_C32", "C33", "D", "E", "E36", "E37-E39", "F", "G", "G-U_X_H", "G45", "G46", "G47", "H", "H49", "H50", "H51", "H52", "H53", "I", "J", "J58", "J59_J60", "J61", "J62_J63", "K", "K64", "K65", "K66", "L", "L68A", "M", "M69_M70", "M71", "M72", "M73", "M74_M75", "N", "N77", "N78", "N79", "N80-N82", "O", "P", "Q", "Q86", "Q87_Q88", "R", "R90-R92", "R93", "S", "S94", "S95", "S96", "T", "TOTAL", "U"]

params = {
    "geo": geo,
    "time": time,
    "unit": unit,
    "airpol": airpol,
    "nace_r2": nace_r2,
    "na_item": na_item,
    "format": "JSON"
}

req_eurostat = req.get(url_eurostat, params=params, timeout=30)
eurostat = req_eurostat.json()
print(req_eurostat.status_code)

200


Eurostat non restituisce una tabella pronta all'uso, bensì una struttura JSON-stat compressa: le combinazioni settore/anno sono codificate come posizioni numeriche all'interno di un unico dizionario "value". Di seguito si ricostruisce la tabella tramite un ciclo esplicito.

In [2]:
nace_r2_values = []
time_values = []
ghg_values = []

for settore in nace_r2:
    indice_settore = eurostat["dimension"]["nace_r2"]["category"]["index"][settore]
    for indice_anno in range(0, len(time)):
        posizione = indice_settore * len(time) + indice_anno
        ghg_values.append(eurostat["value"].get(str(posizione)))
        nace_r2_values.append(settore)
        time_values.append(time[indice_anno])

dataset = {
    "NACE": nace_r2_values,
    "Anno": time_values,
    "GHG": ghg_values
}

dataset_df = pd.DataFrame(dataset)
dataset_df["Fonte"] = "eurostat"

print(dataset_df.shape)
print(dataset_df["GHG"].isna().sum())

(1264, 4)
16


In [3]:
# Correzione outlier C19: valore aggiunto settoriale negativo in 2014 e 2020
# genera un'intensità non interpretabile economicamente. Sostituito con interpolazione lineare (media dei due anni adiacenti).

for anno_anomalo, anno_prima, anno_dopo in [("2014", "2013", "2015"), ("2020", "2019", "2021")]:
    valore_prima = dataset_df[(dataset_df["NACE"] == "C19") & (dataset_df["Anno"] == anno_prima)]["GHG"].values[0]
    valore_dopo = dataset_df[(dataset_df["NACE"] == "C19") & (dataset_df["Anno"] == anno_dopo)]["GHG"].values[0]
    valore_interpolato = (valore_prima + valore_dopo) / 2

    indice = dataset_df[(dataset_df["NACE"] == "C19") & (dataset_df["Anno"] == anno_anomalo)].index
    dataset_df.loc[indice, "GHG"] = valore_interpolato

print(dataset_df[dataset_df["NACE"] == "C19"][["Anno", "GHG"]].sort_values("Anno"))

     Anno        GHG
176  2008  13985.190
177  2009  21534.060
178  2010  22113.230
179  2011  17291.290
180  2012  17949.400
181  2013  18648.320
182  2014  13421.105
183  2015   8193.890
184  2016  12557.590
185  2017  21312.480
186  2018  47036.600
187  2019  44885.280
188  2020  27388.530
189  2021   9891.780
190  2022   4658.730
191  2023   6607.650


## 2. Banca d'Italia – prestiti per settore e regione

Tavola storica, classificazione Ateco 2007 (non Ateco 2025 - versione differente da quella utilizzata per l'estrazione di un singolo trimestre). Frequenza trimestrale dal 2011 al 2025, successivamente disponibile solo a fine anno.

La colonna "Localizzazione della controparte" include sia le regioni sia le macro-aree (Italia nord-occidentale, ecc.) oltre al totale Italia: tali voci devono essere escluse per evitare un doppio conteggio. La medesima criticità si presenta per la colonna "Attività economica", che contiene sia aggregati (attività manifatturiera, servizi di informazione e comunicazione, totale Ateco) sia le relative sottocategorie.

In [4]:
path = "C:/Users/giovanni.esposito/Desktop/EPICODE/MX CAPSTONE PROJECT/Dataset/Banca_Italia_prestiti_regione_settore_seriestorica.csv"
dataset_bds = pd.read_csv(path, sep=';')

da_escludere = ["Italia", "Italia nord-occidentale", "Italia nord-orientale", "Italia centrale", "Italia meridionale", "Italia insulare"]
dataset_bds_clean = dataset_bds[~dataset_bds["Localizzazione della controparte"].isin(da_escludere)]

settori_da_escludere = [
    "attività manifatturiera",
    "Serv.di informaz.e comunicazione",
    "Totale ateco al netto della sez. U"
]
dataset_bds_clean = dataset_bds_clean[
    ~dataset_bds_clean["Attività economica della controparte (ateco 2007)"].isin(settori_da_escludere)
].copy()

print(dataset_bds_clean.shape)
print(dataset_bds_clean["Attività economica della controparte (ateco 2007)"].nunique())

(34790, 8)
29


In [5]:
# Manteniamo solo il dato di fine anno (31/12), scartando gli altri 3 trimestri
filtro_periodo = dataset_bds_clean["Data dell'osservazione"].str.contains("31/12")
dataset_bds_clean = dataset_bds_clean[filtro_periodo].copy()

colonnasplit = dataset_bds_clean["Data dell'osservazione"].str.split("/")
dataset_bds_clean["Anno"] = colonnasplit.str[2].astype(int)

## 3. Raccordo Ateco 2007 (BDS) – NACE (Eurostat)

Le due classificazioni non presentano una corrispondenza perfetta. Il caso più critico riguarda la categoria residuale BDS "divisioni 16, 32, 33", a fronte di un codice Eurostat che aggrega le divisioni 31 e 32 (mobili e altre manifatture) in un'unica voce, C31_C32. In assenza di un raccordo esatto, si è scelto di attribuire C31_C32 alla voce "Fabbric.mobili" e di mantenere nel residuale solo le divisioni 16 e 33, escludendo la 32 per evitarne il doppio conteggio. Tale limitazione metodologica viene esplicitamente dichiarata anziché omessa.

Nei casi in cui un'etichetta BDS corrisponda a più codici NACE, si adotta la media semplice delle relative intensità emissive, senza ponderazione per valore aggiunto: un affinamento in tal senso non è ritenuto proporzionato rispetto all'obiettivo del progetto.

In [6]:
mapping_ateco_nace = {
    "Agricoltura,silvicoltura e pesca": ["A"],
    "Attiv.dei serv.di alloggio e ristoraz.": ["I"],
    "Attiv.finanziarie e assicurative": ["K"],
    "Attiv.professionali,scientifiche e tecniche": ["M"],
    "Attività immobiliari": ["L"],
    "Attività manifatturiera residuale (divisioni 16,32,33)": ["C16", "C33"],
    "Attività residuali (sezioni O P Q R S T)": ["O", "P", "Q", "R", "S", "T"],
    "Carta, articoli di carta e prodotti della stampa": ["C17", "C18"],
    "Commerc.ingrosso e al dettaglio;riparaz.di autoveicoli e motocicli": ["G"],
    "Costruzioni": ["F"],
    "Estraz.di minerali da cave e miniere": ["B"],
    "Fabbr.computer/prod.elettron./ottica;apparec.elettromed.,apparec.misuraz/orologi": ["C26"],
    "Fabbric.altri prod.della lavoraz.minerali non metalliferi": ["C23"],
    "Fabbric.apparecch.elettriche e apparecch.per uso domest.non elettriche": ["C27"],
    "Fabbric.articoli in gomma e materie plastiche": ["C22"],
    "Fabbric.coke e prod.derivanti dalla raffinaz.del petrolio": ["C19"],
    "Fabbric.macchinari e apparecch.nca": ["C28"],
    "Fabbric.mobili": ["C31_C32"],
    "Fabbric.prod.in metallo,esclusi macchinari e attrezzature": ["C25"],
    "Fabbricazione di autoveicoli e altri mezzi di trasporto": ["C29", "C30"],
    "Fornit.di acqua;reti fognarie,attiv.di gestione dei rifiuti e risanam.": ["E"],
    "Fornit.di energia elettrica,gas,vapore e aria condizionata": ["D"],
    "Industrie alimentari, delle bevande e del tabacco": ["C10-C12"],
    "Industrie tessili,  abbigliamento e articoli in pelle": ["C13-C15"],
    "Metallurgia": ["C24"],
    "Noleggio,agenzie di viaggio,serv.di supporto alle imprese": ["N"],
    "Prodotti chimici e farmaceutici": ["C20", "C21"],
    "Telecomunicazioni": ["J61"],
    "Trasporto e magazzinaggio": ["H"],
}

righe_mappate = []
for etichetta_bds, codici_nace in mapping_ateco_nace.items():
    sottoinsieme = dataset_df[dataset_df["NACE"].isin(codici_nace)]
    intensita_media_per_anno = sottoinsieme.groupby("Anno")["GHG"].mean()
    for anno, valore in intensita_media_per_anno.items():
        righe_mappate.append({
            "settore_bds": etichetta_bds,
            "codici_nace": ",".join(codici_nace),
            "Anno": anno,
            "intensita_ghg": valore,
            "n_codici_nace_aggregati": len(codici_nace)
        })

mapping_intensita = pd.DataFrame(righe_mappate)
mapping_intensita["Anno"] = mapping_intensita["Anno"].astype(int)
print(mapping_intensita.shape)

(464, 5)


## 4. Join e calcolo del TRS

Si procede all'unione tra BDS e la tabella di raccordo, troncando la serie al 2023: Eurostat non copre gli anni successivi, pertanto mantenere i dati BDS 2024-2025 privi di un'intensità emissiva corrispondente non sarebbe informativo.

In [7]:
dataset_finale = dataset_bds_clean.merge(
    mapping_intensita,
    left_on=["Attività economica della controparte (ateco 2007)", "Anno"],
    right_on=["settore_bds", "Anno"],
    how="left"
)

dataset_finale = dataset_finale[dataset_finale["Anno"] <= 2023]

print(dataset_finale.shape)
print(dataset_finale["intensita_ghg"].isna().sum())

(7539, 13)
0


TRS relativo: quota di credito che il singolo settore rappresenta sul totale della regione, moltiplicata per l'intensità emissiva del settore. L'indicatore esprime il grado di sbilanciamento del portafoglio regionale, non l'entità in valore assoluto dell'esposizione.

TRS assoluto: valore del credito moltiplicato per l'intensità emissiva, senza normalizzazione. L'indicatore individua dove si concentra il volume di rischio in termini monetari, indipendentemente dal peso percentuale sul portafoglio regionale.

Si è scelto di mantenere entrambi gli indicatori, in quanto rispondono a due quesiti distinti: composizione del portafoglio vs scala dell'esposizione.

In [8]:
totale_per_regione_anno = dataset_finale.groupby(["Localizzazione della controparte", "Anno"])["Valore"].sum().reset_index()
totale_per_regione_anno = totale_per_regione_anno.rename(columns={"Valore": "totale_prestiti_regione"})

dataset_finale = dataset_finale.merge(
    totale_per_regione_anno,
    on=["Localizzazione della controparte", "Anno"],
    how="left"
)

dataset_finale["quota_settore_regione"] = dataset_finale["Valore"] / dataset_finale["totale_prestiti_regione"]
dataset_finale["TRS_relativo"] = dataset_finale["quota_settore_regione"] * dataset_finale["intensita_ghg"]
dataset_finale["TRS_assoluto"] = dataset_finale["Valore"] * dataset_finale["intensita_ghg"]

dataset_finale[["Localizzazione della controparte", "Anno", "settore_bds", "quota_settore_regione", "intensita_ghg", "TRS_relativo", "TRS_assoluto"]].head(10)

,Localizzazione della controparte,Anno,settore_bds,quota_settore_regione,intensita_ghg,TRS_relativo,TRS_assoluto
0,Piemonte,2023,"Agricoltura,silvicoltura e pesca",0.069075,1327.81,91.718608,4.321259e+09
1,Piemonte,2023,Estraz.di minerali da cave e miniere,0.001924,1795.70,3.455577,1.628071e+08
2,Piemonte,2023,"Fornit.di energia elettrica,gas,vapore e aria ...",0.020079,3564.92,71.579474,3.372418e+09
3,Piemonte,2023,"Fornit.di acqua;reti fognarie,attiv.di gestion...",0.013596,1371.60,18.648121,8.785935e+08
4,Piemonte,2023,Costruzioni,0.085333,74.38,6.347064,2.990376e+08
5,Piemonte,2023,Commerc.ingrosso e al dettaglio;riparaz.di aut...,0.160853,57.03,9.173465,4.322015e+08
6,Piemonte,2023,Trasporto e magazzinaggio,0.031136,501.13,15.603244,7.351362e+08
7,Piemonte,2023,Attiv.dei serv.di alloggio e ristoraz.,0.027877,24.46,0.681880,3.212633e+07
8,Piemonte,2023,Attiv.finanziarie e assicurative,0.008180,6.74,0.055132,2.597515e+06
9,Piemonte,2023,Attività immobiliari,0.066879,1.54,0.102994,4.852508e+06


Prima di considerare attendibile il risultato, si eseguono due controlli: le quote settoriali per regione/anno devono sommare a 1 (in caso contrario il denominatore risulterebbe errato), e l'intensità emissiva di un settore deve presentare variazione nel tempo (in caso contrario il join avrebbe agganciato l'anno errato).

In [9]:
verifica_quote = dataset_finale.groupby(["Localizzazione della controparte", "Anno"])["quota_settore_regione"].sum()
print(verifica_quote.describe())

verifica_variazione = dataset_finale[
    dataset_finale["settore_bds"] == "Metallurgia"
][["Anno", "intensita_ghg"]].drop_duplicates().sort_values("Anno")
print(verifica_variazione)

count    260.0
mean       1.0
std        0.0
min        1.0
25%        1.0
50%        1.0
75%        1.0
max        1.0
Name: quota_settore_regione, dtype: float64
     Anno  intensita_ghg
370  2011        3396.50
341  2012        2817.17
312  2013        2124.75
283  2014        1841.84
254  2015        1616.31
225  2016        1547.86
196  2017        1368.36
167  2018        1728.21
138  2019        1682.54
109  2020        1545.30
80   2021        2199.26
51   2022        2111.07
22   2023        1787.31


## 5. Costruzione dello schema relazionale

Il dataset risultante dal join è strutturato per l'analisi ma non per la visualizzazione: contiene colonne ridondanti (totali di regione, quote intermedie) e ripete a ogni riga attributi che appartengono a livello di settore o di regione, non alla singola osservazione. Si procede pertanto alla scomposizione in uno schema a stella, separando le tabelle di dimensione (settore, regione, tempo) dalla tabella dei fatti, in preparazione al caricamento in Power BI.

La tabella dimensionale dei settori viene arricchita con l'indicatore binario di appartenenza ai settori che contribuiscono in misura considerevole ai cambiamenti climatici, secondo la definizione di cui al considerando 6 del regolamento delegato (UE) 2020/1818, ripresa sia dall'EBA GL 2025/01 sia dal regolamento (UE) 2022/2453: sezioni da A a H e sezione L dell'allegato I del regolamento (CE) n. 1893/2006. L'attribuzione del flag esclude i settori afferenti alle sezioni I, J, K, M, N e ai residuali O-T.

Le quote settoriali e gli altri rapporti calcolati sul totale regionale non vengono replicati come colonne fisiche, poiché la loro correttezza dipende dal contesto di filtro applicato in fase di analisi: la relativa logica viene demandata alle misure DAX in Power BI.

In [13]:
#fact:credito per la tabella principale facts
fact_credito_emissioni = dataset_finale[[
    "Localizzazione della controparte",
    "Anno",
    "settore_bds",
    "Valore",
    "intensita_ghg",
    "TRS_relativo",
    "TRS_assoluto"
]].copy()

fact_credito_emissioni = fact_credito_emissioni.rename(columns={
    "Localizzazione della controparte": "Regione",
    "settore_bds": "Settore"
})

print(fact_credito_emissioni.shape)

#dim_settore per i settori ateco/nace
settori_esclusi_da_flag = [
    "Attiv.dei serv.di alloggio e ristoraz.",
    "Attiv.finanziarie e assicurative",
    "Attiv.professionali,scientifiche e tecniche",
    "Noleggio,agenzie di viaggio,serv.di supporto alle imprese",
    "Telecomunicazioni",
    "Attività residuali (sezioni O P Q R S T)"
]

righe_mappate2 = []
for etichetta_bds, codici_nace in mapping_ateco_nace.items():
    contributo_climatico = etichetta_bds not in settori_esclusi_da_flag
    righe_mappate2.append({
        "settore_bds": etichetta_bds,
        "codici_nace": ",".join(codici_nace),
        "n_codici_nace_aggregati": len(codici_nace),
        "alto_contributo_climatico": contributo_climatico
    })

dim_settore = pd.DataFrame(righe_mappate2)

# Sezione NACE 
sezioni_nace = []
for etichetta_bds, codici_nace in mapping_ateco_nace.items():
    primo_codice = codici_nace[0]
    sezione = primo_codice[0]
    sezioni_nace.append({"settore_bds": etichetta_bds, "sezione_nace": sezione})

sezioni_df = pd.DataFrame(sezioni_nace)
dim_settore = dim_settore.merge(sezioni_df, on="settore_bds", how="left")

etichette_sezioni_nace = {
    "A": "A - Agricoltura, silvicoltura e pesca",
    "B": "B - Estrazione di minerali",
    "C": "C - Attività manifatturiere",
    "D": "D - Energia elettrica, gas, vapore",
    "E": "E - Acqua, reti fognarie, rifiuti",
    "F": "F - Costruzioni",
    "G": "G - Commercio",
    "H": "H - Trasporto e magazzinaggio",
    "I": "I - Alloggio e ristorazione",
    "J": "J - Telecomunicazioni",
    "K": "K - Attività finanziarie e assicurative",
    "L": "L - Attività immobiliari",
    "M": "M - Attività professionali e scientifiche",
    "N": "N - Noleggio e supporto alle imprese",
    "O": "O-T - Attività residuali"
}

dim_settore["sezione_nace_label"] = dim_settore["sezione_nace"].map(etichette_sezioni_nace)

print(dim_settore.shape)
print(dim_settore["sezione_nace_label"].isna().sum())

#dim_regione per le regioni italiane
dim_regione = pd.DataFrame(dataset_finale["Localizzazione della controparte"].unique())
dim_regione = dim_regione.rename(columns={0: "Regione"})

# dim_tempo per gli anni
dim_tempo = pd.DataFrame(dataset_finale["Anno"].unique())
dim_tempo = dim_tempo.rename(columns={0: "Anno"})

(7539, 7)
(29, 6)
0


## 6. Esportazione dei dati

Le quattro tabelle dello schema a stella vengono esportate in formato CSV con separatore punto-e-virgola, coerente con la convention italiana e con i file di origine. I CSV costituiscono il formato di transito verso Power BI per la costruzione della model relazionale e delle relative visualizzazioni.

In [14]:
fact_credito_emissioni.to_csv(
    "fact_credito_emissioni.csv",
    index=False,
    sep=';',
    decimal=',',
    float_format='%.6f',
    encoding='utf-8-sig'
)
dim_settore.to_csv("dim_settore.csv", index=False, sep=';', encoding='utf-8')
dim_regione.to_csv("dim_regione.csv", index=False, sep=';', encoding='utf-8-sig')
dim_tempo.to_csv("dim_tempo.csv", index=False, sep=';', encoding='utf-8')

print("Export completato")

Export completato
